<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/plant_pathology_KD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
!mv PlantDoc-Dataset /content/PlantDoc
!ls /content/PlantDoc

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 50.08 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
!unzip Plant_Pathology2020.zip

Archive:  Plant_Pathology2020.zip
   creating: Plant Pathology 2020/
   creating: Plant Pathology 2020/Images/
   creating: Plant Pathology 2020/Images/healthy/
  inflating: Plant Pathology 2020/Images/healthy/Train_100.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1001.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1002.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1004.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1005.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1007.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1012.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1014.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1017.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1020.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_103.jpg  
  inflating: Plant Pathology 2020/Images/healthy/Train_1031.jpg  
  inflating: Plant Pathology 2020/Images/healthy/

In [ ]:
# =======================
# 1. Imports
# =======================
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import EfficientNetB4
from sklearn.metrics import classification_report, accuracy_score

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [ ]:
# =======================
# 2. Paths
# =======================
teacher_model_path = "/content/best_model.h5"
pp_root = "/content/Plant Pathology 2020/Images"   # healthy / rust / scab
plantdoc_train = "/content/PlantDoc/train"

In [ ]:
# =======================
# 3. Load Teacher Model
# =======================
teacher_model = load_model(teacher_model_path)
teacher_model.trainable = False
print("Teacher model loaded.")

Teacher model loaded.


In [ ]:
# =======================
# 4. Load Plant Pathology Dataset
# =======================
img_size = (380, 380)
batch_size = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    pp_root,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    validation_split=0.2,
    subset="training",
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    pp_root,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    validation_split=0.2,
    subset="validation",
    seed=42
)

class_names_apple = train_ds.class_names
print("Plant Pathology classes:", class_names_apple)

Found 1730 files belonging to 3 classes.
Using 1384 files for training.
Found 1730 files belonging to 3 classes.
Using 346 files for validation.
Plant Pathology classes: ['healthy', 'rust', 'scab']


In [ ]:
# =======================
# 5. Data Augmentation
# =======================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y)).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
# =======================
# 6. Student Model
# =======================
student_base = EfficientNetB4(
    include_top=False,
    input_shape=(380, 380, 3),
    weights="imagenet"
)

student_model = tf.keras.Sequential([
    student_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation="softmax")
])

print("Student model created.")

71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Student model created.


In [ ]:
# =======================
# 7. PlantDoc Class Names
# =======================
def normalize_name(n):
    return n.lower().replace("_", " ").replace("-", " ").strip()

class_names_plantdoc = [
    c for c in os.listdir(plantdoc_train)
    if os.path.isdir(os.path.join(plantdoc_train, c))
]

class_names_plantdoc_norm = [normalize_name(c) for c in class_names_plantdoc]
print("PlantDoc classes:", class_names_plantdoc_norm)

PlantDoc classes: ['peach leaf', 'blueberry leaf', 'bell pepper leaf', 'potato leaf late blight', 'tomato early blight leaf', 'apple leaf', 'tomato leaf late blight', 'strawberry leaf', 'raspberry leaf', 'cherry leaf', 'soyabean leaf', 'potato leaf early blight', 'corn gray leaf spot', 'corn leaf blight', 'tomato leaf bacterial spot', 'tomato two spotted spider mites leaf', 'apple rust leaf', 'bell pepper leaf spot', 'tomato leaf mosaic virus', 'tomato leaf yellow virus', 'apple scab leaf', 'corn rust leaf', 'tomato leaf', 'tomato septoria leaf spot', 'grape leaf', 'squash powdery mildew leaf', 'tomato mold leaf', 'grape leaf black rot']


In [ ]:
# =======================
# 8. Mapping: PP → PlantDoc
# =======================
mapping = {
    "healthy": "Apple leaf",
    "rust": "Apple rust leaf",
    "scab": "Apple Scab Leaf"
}

relevant_indices = [
    class_names_plantdoc_norm.index(normalize_name(mapping[c]))
    for c in class_names_apple
]

print("Relevant teacher indices:", relevant_indices)

Relevant teacher indices: [5, 16, 20]


In [ ]:
# =======================
# 9. Distillation Setup
# =======================
T = 5.0
alpha = 0.7

optimizer = tf.keras.optimizers.Adam(1e-4)
ce_loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:
        # Teacher
        teacher_logits = teacher_model(images, training=False)
        teacher_probs = tf.nn.softmax(teacher_logits / T)
        teacher_probs_relevant = tf.gather(teacher_probs, relevant_indices, axis=1)

        # Student
        student_logits = student_model(images, training=True)
        student_probs = tf.nn.softmax(student_logits / T)

        # Normalize
        teacher_probs_relevant /= tf.reduce_sum(teacher_probs_relevant, axis=1, keepdims=True)
        student_probs /= tf.reduce_sum(student_probs, axis=1, keepdims=True)

        eps = 1e-7
        teacher_probs_relevant = tf.clip_by_value(teacher_probs_relevant, eps, 1.0)
        student_probs = tf.clip_by_value(student_probs, eps, 1.0)

        # Symmetric KL
        kl_forward = tf.reduce_sum(
            teacher_probs_relevant * tf.math.log(teacher_probs_relevant / student_probs), axis=1
        )
        kl_backward = tf.reduce_sum(
            student_probs * tf.math.log(student_probs / teacher_probs_relevant), axis=1
        )

        kl_loss = tf.reduce_mean(0.5 * (kl_forward + kl_backward))
        soft_loss = kl_loss * (T ** 2)
        hard_loss = ce_loss_fn(labels, student_logits)

        loss = alpha * soft_loss + (1.0 - alpha) * hard_loss

    grads = tape.gradient(loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, student_model.trainable_variables))
    return tf.reduce_mean(loss)

In [ ]:
# =======================
# 10. Training
# =======================
epochs = 5

for epoch in range(epochs):
    total_loss = 0.0
    steps = 0

    for step, (images, labels) in enumerate(train_ds):
        loss = train_step(images, labels)
        total_loss += float(loss)
        steps += 1

        if step % 100 == 0:
            tf.print(f"Epoch {epoch+1}, Step {step}, Loss:", loss)

    avg_loss = total_loss / steps
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

Epoch 1, Step 0, Loss: 0.334807456
Epoch 1, Step 100, Loss: 0.154573813
Epoch 1/5 - Avg Loss: 0.2064
Epoch 2, Step 0, Loss: 0.15152505
Epoch 2, Step 100, Loss: 0.159671575
Epoch 2/5 - Avg Loss: 0.1693
Epoch 3, Step 0, Loss: 0.153076053
Epoch 3, Step 100, Loss: 0.151011363
Epoch 3/5 - Avg Loss: 0.1622
Epoch 4, Step 0, Loss: 0.155297056
Epoch 4, Step 100, Loss: 0.150769606
Epoch 4/5 - Avg Loss: 0.1597
Epoch 5, Step 0, Loss: 0.151590288
Epoch 5, Step 100, Loss: 0.147548437
Epoch 5/5 - Avg Loss: 0.1548


In [ ]:
# =======================
# 11. Evaluation
# =======================
y_true, y_pred = [], []

for images, labels in test_ds:
    preds = student_model.predict(images, verbose=0)
    preds = np.argmax(preds, axis=1)
    y_pred.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

print("\nFINAL EVALUATION:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names_apple))


FINAL EVALUATION:
Accuracy: 0.9797687861271677
              precision    recall  f1-score   support

     healthy       0.95      0.99      0.97        94
        rust       0.99      1.00      1.00       118
        scab       0.99      0.96      0.97       134

    accuracy                           0.98       346
   macro avg       0.98      0.98      0.98       346
weighted avg       0.98      0.98      0.98       346



In [ ]:
# =======================
# 12. Save & Download
# =======================
student_model.save("/content/student_model_pp2020.h5")
print("Student model saved.")

model_size = os.path.getsize("/content/student_model_pp2020.h5") / (1024 * 1024)
print(f"Model size: {model_size:.2f} MB")

from google.colab import files
files.download("/content/student_model_pp2020.h5")

Student model saved.
Model size: 68.31 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>